# DAY 2 — Feature Engineering · 모델 개선 · 결과 해석

## 실습 운영 안내
- 수업 시간 내 완료하지 못한 실습은 과제로 수행

## 목차
24. 추가 Feature Engineering  
25. 모델링 데이터 준비  
26. Feature Engineering 이후 모델 재학습  
27. DAY 1과 Feature Engineering 이후 성능 비교  
28. 전처리 방법 변경 실험  
29. 머신러닝 Hyperparameter 변경  
30. Keras 모델 구조 변경  
31. Keras Hyperparameter 변경  
32. 실험 결과 정리 및 비교  
33. Overfitting 확인  
34. Feature Importance 분석  
35. Actual vs Predicted / Residual 분석  
36. 오차가 큰 데이터 분석  
37. 최종 모델 선정  
38. Wrap-up

## 쉬운 풀이 버전 사용법
- 코드는 `# 모델 불러오기`, `# 모델 생성`, `# 모델 학습`, `# 예측`, `# 평가지표 비교`처럼 단계별 주석을 달았습니다.
- 위에서부터 순서대로 실행하면 됩니다.
- 실제 실행 결과가 있어야 정할 수 있는 답은 숫자를 임의로 만들지 않고, **어떤 결과를 보고 작성하는지** 적었습니다.
- MAE / RMSE는 **작을수록 좋고**, R²는 **1에 가까울수록 좋습니다.**


## DAY 1에서 준비할 결과물

DAY 2 시작 전 DAY 1 Notebook에서 다음 두 파일을 저장해 둡니다.

**EDA 및 기본 파생변수가 포함된 데이터**
```python
df.to_csv("SeoulBikeData_day1_processed.csv", index=False)
```

실험 결과는 별도 표 또는 Excel에 계속 기록합니다.

| Day | Encoding | Scaling | Feature | Model | 변경 내용 | MAE | RMSE | R² |
|---|---|---|---|---|---|---:|---:|---:|
| DAY1 | One-Hot | Standard | 기본 | Random Forest | 기본 | | | |
| DAY2 | One-Hot | Standard | 추가 FE | Random Forest | Feature 추가 | | | |
| DAY2 | One-Hot | MinMax | 추가 FE | Keras | Scaling 변경 | | | |
| DAY2 | Ordinal | Standard | 추가 FE | Random Forest | Encoding 변경 | | | |

## 0. DAY 2 공통 준비

In [1]:
# 필요한 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Train / Test 분리
from sklearn.model_selection import train_test_split

# Scaling / Encoding 도구 불러오기
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OrdinalEncoder

# 머신러닝 모델 불러오기
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 회귀 평가지표 불러오기
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Keras 불러오기
from tensorflow import keras

In [2]:
# DAY1에서 저장한 데이터 불러오기
df = pd.read_csv("SeoulBikeData_day1_processed.csv")

# date 컬럼을 날짜 형식으로 변경
df["date"] = pd.to_datetime(df["date"])

# 데이터 확인
df.head()

In [3]:
# MAE, RMSE, R2를 한 번에 계산하는 함수 만들기
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

# STEP 24. 추가 Feature Engineering

### Q1. DAY 1에서 확인한 오전·오후 출퇴근 패턴을 어떻게 Feature로 표현할 수 있을까?

**답:**

In [7]:
# 오전 출퇴근 시간: 7시, 8시, 9시면 1 / 아니면 0
df["morning_rush"] = df["hour"].isin([7, 8, 9]).astype(int)

# 오후 출퇴근 시간: 17시, 18시, 19시면 1 / 아니면 0
df["evening_rush"] = df["hour"].isin([17, 18, 19]).astype(int)

### Q2. 이미 `hour`가 있는데 `morning_rush`, `evening_rush`를 추가하는 것은 중복 아닐까?

**답:**

완전히 같은 정보는 아니다. `hour`는 시간 자체를 나타내고, `morning_rush`, `evening_rush`는 출퇴근 시간인지 바로 알려주는 Feature이다. 모델이 출퇴근 시간의 특징을 더 쉽게 학습하는 데 도움을 줄 수 있다.


### Q3. 비가 오는 출퇴근 시간을 별도의 Feature로 표현할 수 있을까?

**답:**

In [8]:
# 비가 오면서 오전 또는 오후 출퇴근 시간이면 1, 아니면 0
df["rain_rush"] = (
    (df["is_rain"] == 1) &
    ((df["morning_rush"] == 1) | (df["evening_rush"] == 1))
).astype(int)

### Q4. 이미 각각의 Feature가 있는데 `rain_rush`까지 추가할 필요가 있을까?

**답:**

`is_rain`과 출퇴근 Feature를 따로 사용하는 것과 달리, `rain_rush`는 '비 오는 출퇴근 시간'이라는 두 조건의 조합을 직접 표현한다. 이런 조합이 대여량에 영향을 준다면 도움이 될 수 있다.


### Q5. 새로 만든 Feature가 의도한 대로 생성되었는지 확인해보자.

**답:**

In [14]:
# 새로 만든 출퇴근 Feature 확인
df[["hour", "morning_rush", "evening_rush", "rain_rush"]].head(24)

,hour,morning_rush,evening_rush,rain_rush
0,0,0,0,False
1,1,0,0,False
2,2,0,0,False
3,3,0,0,False
4,4,0,0,False
5,5,0,0,False
6,6,0,0,False
7,7,1,0,False
8,8,1,0,False
9,9,1,0,False


### Q6. 23시와 0시는 실제로 가까운 시간인데 숫자로는 멀리 떨어져 있다. 이를 어떻게 표현할 수 있을까?

**답:**

In [18]:
# 24시간을 원처럼 표현하기 위해 sin 값 생성
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)

# cos 값도 함께 생성
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# 결과 확인
df[["hour", "hour_sin", "hour_cos"]].head(24)

,hour,hour_sin,hour_cos
0,0,0.000000e+00,1.000000e+00
1,1,2.588190e-01,9.659258e-01
2,2,5.000000e-01,8.660254e-01
3,3,7.071068e-01,7.071068e-01
4,4,8.660254e-01,5.000000e-01
5,5,9.659258e-01,2.588190e-01
6,6,1.000000e+00,6.123234e-17
7,7,9.659258e-01,-2.588190e-01
8,8,8.660254e-01,-5.000000e-01
9,9,7.071068e-01,-7.071068e-01


### 생각해보기
- Linear Regression에서는 이런 표현이 도움이 될까?
- Random Forest에서도 같은 정도로 필요할까?

**답:**

Linear Regression은 숫자 사이의 선형 관계를 주로 보기 때문에 `hour_sin`, `hour_cos`처럼 시간의 반복 구조를 표현하면 도움이 될 수 있다. Random Forest는 시간에 따른 복잡한 관계를 스스로 나눠 학습할 수 있어 Linear Regression보다는 필요성이 작을 수 있다.


### 추가 Feature가 포함된 데이터 저장

In [ ]:
# DAY2에서 추가한 Feature가 포함된 데이터 저장
df.to_csv("SeoulBikeData_day2_feature.csv", index=False)

# STEP 25. 모델링 데이터 준비

### Q7. 어떤 컬럼에 Encoding이 필요할까?

**답:**

In [20]:
# 각 컬럼의 데이터 타입 확인
df.dtypes

rented                          int64
hour                            int64
temp                          float64
humidity                        int64
wind                          float64
visibility                      int64
dew_point                     float64
solar                         float64
rainfall                      float64
snowfall                      float64
weekday                         int64
month                           int64
is_weekend                      int64
is_rain                         int64
is_snow                         int64
time_group                     object
season_Autumn                   int64
season_Spring                   int64
season_Summer                   int64
season_Winter                   int64
holiday_Holiday                 int64
holiday_No Holiday              int64
functioning_day_No              int64
functioning_day_Yes             int64
day_name_Friday                 int64
day_name_Monday                 int64
day_name_Sat

In [22]:
# 문자열(object) 컬럼만 찾기
obj_cols = df.select_dtypes(include="object").columns

# Encoding이 필요한 컬럼 확인
print(obj_cols)

Index(['time_group'], dtype='object')


### Q8. Encoding은 Train/Test 분리 전과 후 중 언제 하는 것이 좋을까?

다음 두 방법의 차이를 생각해보자.

**방법 1**  
전체 데이터 → Encoding → Train/Test Split

**방법 2**  
Train/Test Split → Train 데이터로 Encoder 학습 → Train/Test Encoding

### 생각해보기
- 두 방법은 어떤 차이가 있을까?
- Test 데이터에 새로운 범주가 나타날 수 있다면 어떤 방법이 적절할까?
- 이번 데이터에서는 어떤 방식을 사용하는 것이 좋을까?

**답:**

일반적으로는 Train/Test를 먼저 나누고 Train 데이터로 Encoder를 학습한 뒤 Test에 적용하는 방법이 더 안전하다. Test 정보를 미리 사용하면 데이터 누수가 생길 수 있기 때문이다. 다만 이번 실습에서는 DAY1과 비교하기 쉽게 전체 데이터에 One-Hot Encoding을 적용한 뒤 분리하는 방식으로 진행한다.


### One-Hot Encoding

In [ ]:
# date는 날짜형이므로 모델 입력에서 제외
df_model = df.drop(columns=["date"], errors="ignore")

# 문자형 컬럼을 One-Hot Encoding
df_onehot = pd.get_dummies(df_model, dtype=int)

# 변환 결과 확인
df_onehot.head()

### Feature / Target 분리

In [ ]:
# 입력 Feature 만들기: rented를 제외한 나머지 컬럼
X = df_onehot.drop("rented", axis=1)

# Target 만들기: 예측할 자전거 대여량
y = df_onehot["rented"]

# 크기 확인
print(X.shape)
print(y.shape)

### Train / Test Split

In [ ]:
# Train 70%, Test 30%로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=1002
)

# 데이터 크기 확인
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

### Q9. Encoding은 전체 데이터에 먼저 적용했는데 Scaling도 전체 데이터에 먼저 적용하면 될까?

### 생각해보기
- `StandardScaler`는 데이터에서 무엇을 계산할까?
- Test 데이터까지 포함하여 평균과 표준편차를 계산하면 어떤 문제가 생길까?
- 왜 Train에는 `fit_transform()`, Test에는 `transform()`만 사용할까?

**답:**

Scaling은 전체 데이터에 먼저 적용하면 안 된다. StandardScaler는 평균과 표준편차를 계산하므로 Test까지 함께 `fit`하면 Test 정보가 Train 과정에 들어갈 수 있다. 따라서 Train에는 `fit_transform()`, Test에는 Train에서 배운 기준으로 `transform()`만 사용한다.


### 기본 Scaling

In [ ]:
# StandardScaler 생성
scaler = StandardScaler()

# Train 데이터로 평균/표준편차를 학습하면서 Scaling
x_train = scaler.fit_transform(X_train)

# Train에서 학습한 기준으로 Test 데이터 Scaling
x_test = scaler.transform(X_test)

# STEP 26. Feature Engineering 이후 모델 재학습

## 머신러닝 모델

**가이드:** Linear Regression, Decision Tree, Random Forest, Gradient Boosting을 DAY 1과 동일한 설정으로 학습하고 MAE, RMSE, R²를 기록합니다.

In [ ]:
# 모델 불러오기
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 모델 생성
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=1002)
rf = RandomForestRegressor(random_state=1002)
gb = GradientBoostingRegressor(random_state=1002)

# 모델 학습
lr.fit(x_train, y_train)
dt.fit(x_train, y_train)
rf.fit(x_train, y_train)
gb.fit(x_train, y_train)

# 모델별 예측
y_pred_lr = lr.predict(x_test)
y_pred_dt = dt.predict(x_test)
y_pred_rf = rf.predict(x_test)
y_pred_gb = gb.predict(x_test)

# 평가지표 비교
print("Linear Regression :", evaluate(y_test, y_pred_lr))
print("Decision Tree     :", evaluate(y_test, y_pred_dt))
print("Random Forest     :", evaluate(y_test, y_pred_rf))
print("Gradient Boosting :", evaluate(y_test, y_pred_gb))

## Keras Baseline

**가이드:** DAY 1과 동일한 `64 → 32 → 1` 구조, ReLU, Adam, MSE를 사용합니다.

In [ ]:
# Keras 모델 생성
model = keras.Sequential([
    # 첫 번째 은닉층: 입력 Feature를 받아 64개의 뉴런으로 학습
    keras.layers.Dense(64, activation="relu", input_shape=(x_train.shape[1],)),

    # 두 번째 은닉층
    keras.layers.Dense(32, activation="relu"),

    # 회귀 문제이므로 최종 출력값은 1개
    keras.layers.Dense(1)
])

# 모델 학습 방법 설정
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# 모델 학습
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# Test 데이터 예측
y_pred_keras = model.predict(x_test).flatten()

# Keras 성능 확인
print("Keras Baseline :", evaluate(y_test, y_pred_keras))

# STEP 27. DAY 1과 Feature Engineering 이후 성능 비교

In [ ]:
# DAY2 Feature Engineering 이후 결과를 표로 정리
day2_result = pd.DataFrame([
    ["Linear Regression", *evaluate(y_test, y_pred_lr).values()],
    ["Decision Tree", *evaluate(y_test, y_pred_dt).values()],
    ["Random Forest", *evaluate(y_test, y_pred_rf).values()],
    ["Gradient Boosting", *evaluate(y_test, y_pred_gb).values()],
    ["Keras", *evaluate(y_test, y_pred_keras).values()]
], columns=["Model", "MAE", "RMSE", "R2"])

# 결과 확인
day2_result

### Q10. Feature Engineering 이후 모든 모델의 성능이 좋아졌는가?

**답:**

실행 결과를 DAY1 결과와 비교한다. MAE와 RMSE가 작아지고 R²가 커졌다면 성능이 좋아진 것이다. 모든 모델이 반드시 좋아지는 것은 아니다.


### Q11. 가장 큰 성능 변화가 나타난 모델은 무엇인가?

**답:**

DAY1과 DAY2의 MAE, RMSE, R² 차이가 가장 크게 나타난 모델을 확인하여 작성한다. 실제 실행 결과를 보고 결정한다.


### Q12. 같은 Feature를 추가했는데 모델마다 결과가 다르게 나타나는 이유는 무엇일까?

**답:**

모델마다 데이터를 학습하는 방식이 다르기 때문이다. Linear Regression은 선형 관계를 중심으로 학습하고, Tree 계열은 조건을 나누면서 비선형 관계도 학습할 수 있어 같은 Feature를 추가해도 영향이 다를 수 있다.


# STEP 28. 전처리 방법 변경 실험

## 28-1. Scaling 방법 비교

In [ ]:
# 비교할 Scaler 생성
standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

# Standard Scaling
x_train_std = standard_scaler.fit_transform(X_train)
x_test_std = standard_scaler.transform(X_test)

# MinMax Scaling
x_train_mm = minmax_scaler.fit_transform(X_train)
x_test_mm = minmax_scaler.transform(X_test)

# Robust Scaling
x_train_rb = robust_scaler.fit_transform(X_train)
x_test_rb = robust_scaler.transform(X_test)

### Q13. Scaler를 변경하면 Tree 기반 모델의 성능도 크게 달라질까?

### 먼저 예상해보기
- Decision Tree:
- Random Forest:
- Gradient Boosting:

**예상:**

**답:**

**예상:** Decision Tree, Random Forest, Gradient Boosting 모두 Scaler를 바꿔도 성능 변화가 크지 않을 것으로 예상한다.

**답:** 실제 아래 결과를 비교하여 확인한다. Tree 계열은 값의 크기보다 데이터를 어떤 기준으로 나눌지를 중요하게 보기 때문에 Scaling 영향이 작은 편이다.


In [ ]:
# Scaler별 Random Forest 성능을 확인하는 함수
def test_rf_scaler(name, train_data, test_data):
    # Random Forest 모델 생성
    model_rf = RandomForestRegressor(random_state=1002)

    # 모델 학습
    model_rf.fit(train_data, y_train)

    # 예측
    pred = model_rf.predict(test_data)

    # 결과 출력
    print(name, evaluate(y_test, pred))

# Scaling 방법별 비교
test_rf_scaler("Standard", x_train_std, x_test_std)
test_rf_scaler("MinMax", x_train_mm, x_test_mm)
test_rf_scaler("Robust", x_train_rb, x_test_rb)

### Q14. Tree 기반 모델에서는 왜 Scaler를 바꿔도 결과가 거의 변하지 않을까?

### 생각해보기
Scaling 전: `10, 20, 30`  
Scaling 후: `-1.2, 0.0, 1.2`

숫자는 바뀌었지만 무엇은 그대로 유지될까?

**답:**

Scaling을 해도 데이터의 순서는 그대로 유지된다. Tree 모델은 특정 값보다 큰지 작은지를 기준으로 데이터를 나누기 때문에 값의 범위가 바뀌어도 결과가 거의 달라지지 않을 수 있다.


### Q15. 그렇다면 Neural Network도 Scaler의 영향을 거의 받지 않을까?

**예상:**

**답:**

**예상:** Neural Network는 Tree와 달리 Scaler의 영향을 받을 수 있다.

**답:** Neural Network는 입력값을 이용해 가중치를 반복적으로 업데이트하므로 Feature마다 값의 범위가 크게 다르면 학습이 어려워질 수 있다.


In [ ]:
# Keras 모델을 만들어 Scaler별로 비교하는 함수
def test_keras_scaler(name, train_data, test_data):
    # 모델 생성
    keras_model = keras.Sequential([
        keras.layers.Dense(64, activation="relu", input_shape=(train_data.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1)
    ])

    # 모델 설정
    keras_model.compile(optimizer="adam", loss="mse")

    # 모델 학습
    keras_model.fit(
        train_data, y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        verbose=0
    )

    # 예측
    pred = keras_model.predict(test_data, verbose=0).flatten()

    # 결과 출력
    print(name, evaluate(y_test, pred))

# Scaler별 Keras 성능 비교
test_keras_scaler("Standard", x_train_std, x_test_std)
test_keras_scaler("MinMax", x_train_mm, x_test_mm)
test_keras_scaler("Robust", x_train_rb, x_test_rb)

### Q16. Keras에서는 Scaler에 따라 결과가 어떻게 달라졌는가?

### 생각해보기
- Tree 모델과 차이가 있는가?
- Neural Network에서는 왜 입력값의 범위가 학습에 영향을 줄 수 있을까?

**답:**

실행 결과의 MAE, RMSE, R²를 비교해서 가장 좋은 Scaler를 확인한다. Neural Network는 입력값의 범위에 따라 가중치 학습 과정이 달라질 수 있어 Tree 모델보다 Scaling의 영향을 더 받을 수 있다.


### Q17. 이상치가 많은 데이터라면 어떤 Scaler를 우선 고려할 수 있을까?

**답:**

이상치가 많다면 `RobustScaler`를 우선 고려할 수 있다. 중앙값과 사분위수 범위를 이용하기 때문에 이상치의 영향을 비교적 적게 받는다.


## 28-2. Encoding 방법 변경

### Q18. One-Hot Encoding 대신 Ordinal Encoding을 사용하면 어떻게 될까?

예: `Spring → 0, Summer → 1, Autumn → 2, Winter → 3`

### 생각해보기
실제로 크기 관계가 없는 범주에 숫자를 부여하면 모델은 이를 어떻게 받아들일까?

**답:**

Ordinal Encoding은 범주를 0, 1, 2, 3 같은 숫자로 바꾼다. 하지만 실제 크기 순서가 없는 범주에도 숫자의 순서가 생기므로 모델이 존재하지 않는 순서 관계로 받아들일 수 있다.


In [ ]:
# Ordinal Encoding을 실험하기 위해 원본 모델링 데이터 복사
df_ordinal = df_model.copy()

# 문자형 컬럼 찾기
ordinal_cols = df_ordinal.select_dtypes(include="object").columns.tolist()

# OrdinalEncoder 생성
ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# 문자형 컬럼을 숫자로 변환
df_ordinal[ordinal_cols] = ordinal_encoder.fit_transform(df_ordinal[ordinal_cols])

# 결과 확인
df_ordinal.head()

In [ ]:
# Feature / Target 분리
X_ord = df_ordinal.drop("rented", axis=1)
y_ord = df_ordinal["rented"]

# Train / Test 분리
X_train_ord, X_test_ord, y_train_ord, y_test_ord = train_test_split(
    X_ord, y_ord,
    test_size=0.3,
    random_state=1002
)

# Random Forest 생성
rf_ord = RandomForestRegressor(random_state=1002)

# 모델 학습
rf_ord.fit(X_train_ord, y_train_ord)

# 예측
pred_ord = rf_ord.predict(X_test_ord)

# Ordinal Encoding 성능 확인
print("Ordinal + Random Forest :", evaluate(y_test_ord, pred_ord))

# One-Hot Encoding 성능도 다시 출력하여 비교
print("One-Hot + Random Forest :", evaluate(y_test, y_pred_rf))

### Q19. One-Hot과 Ordinal Encoding에서 성능은 어떻게 달라졌는가?

**답:**

위 결과에서 MAE와 RMSE는 작은 쪽, R²는 큰 쪽이 더 좋은 성능이다. One-Hot과 Ordinal 중 실제 실행 결과가 더 좋은 방법을 확인해서 작성한다.


### Q20. 성능이 더 좋았다는 이유만으로 해당 Encoding이 항상 더 적절하다고 볼 수 있을까?

**답:**

아니다. 성능뿐 아니라 데이터의 의미도 고려해야 한다. 순서가 없는 범주형 데이터에는 일반적으로 One-Hot Encoding이 자연스럽고, 단순히 한 번의 성능이 더 좋았다는 이유만으로 항상 적절하다고 볼 수는 없다.


# STEP 29. 머신러닝 Hyperparameter 변경

### Q21. Random Forest의 `max_depth`를 변경하면 Train/Test 성능은 어떻게 달라질까?

**답:**

`max_depth`가 작으면 모델이 단순해지고 너무 크면 Train 데이터에 지나치게 맞춰질 수 있다. 아래 코드로 Train/Test RMSE를 함께 비교한다.


In [ ]:
# 비교할 max_depth 값
depth_list = [3, 5, 10, None]

# 깊이별 성능 확인
for depth in depth_list:
    # Random Forest 모델 생성
    rf_depth = RandomForestRegressor(
        max_depth=depth,
        random_state=1002
    )

    # 모델 학습
    rf_depth.fit(x_train, y_train)

    # Train / Test 예측
    train_pred = rf_depth.predict(x_train)
    test_pred = rf_depth.predict(x_test)

    # Train / Test RMSE 계산
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    # 결과 출력
    print("max_depth =", depth)
    print("Train RMSE :", train_rmse)
    print("Test RMSE  :", test_rmse)
    print()

### 생각해보기
- 가장 깊은 모델이 항상 좋은가?
- Train RMSE만 보고 모델을 선택해도 될까?

**답:**

가장 깊은 모델이 항상 좋은 것은 아니다. 너무 깊으면 Train 데이터에는 잘 맞지만 Test 성능이 나빠지는 과적합이 발생할 수 있다. 따라서 Train RMSE만 보지 말고 Test RMSE도 함께 확인해야 한다.


### Q22. Gradient Boosting의 `learning_rate`를 변경하면 성능은 어떻게 달라질까?

**답:**

`learning_rate`는 Gradient Boosting이 이전 모델의 오차를 얼마나 크게 보완할지 조절한다. 너무 크거나 너무 작으면 성능이 나빠질 수 있으므로 여러 값을 비교한다.


In [ ]:
# 비교할 learning_rate 값
lr_list = [0.01, 0.05, 0.1, 0.2]

# learning_rate별 성능 비교
for rate in lr_list:
    # Gradient Boosting 모델 생성
    gb_rate = GradientBoostingRegressor(
        learning_rate=rate,
        random_state=1002
    )

    # 모델 학습
    gb_rate.fit(x_train, y_train)

    # 예측
    pred = gb_rate.predict(x_test)

    # 성능 출력
    print("learning_rate =", rate, evaluate(y_test, pred))

# STEP 30. Keras 모델 구조 변경

Baseline 구조: `64 → 32 → 1`

### Q23. Hidden Unit을 늘리면 성능은 좋아질까?

**답:**

Hidden Unit을 늘리면 더 복잡한 관계를 학습할 수 있지만 반드시 성능이 좋아지는 것은 아니다. 실제 Baseline과 Wide Model의 Test 성능을 비교해야 한다.


### Wide Model: `128 → 64 → 1`

In [ ]:
# Wide Model 생성: 128 → 64 → 1
wide_model = keras.Sequential([
    # 첫 번째 은닉층의 뉴런을 128개로 증가
    keras.layers.Dense(128, activation="relu", input_shape=(x_train.shape[1],)),

    # 두 번째 은닉층
    keras.layers.Dense(64, activation="relu"),

    # 출력층
    keras.layers.Dense(1)
])

# 모델 설정
wide_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 모델 학습
wide_history = wide_model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# 예측
wide_pred = wide_model.predict(x_test, verbose=0).flatten()

# 성능 확인
print("Wide Model :", evaluate(y_test, wide_pred))

### Q24. Layer를 하나 더 추가하면 성능은 좋아질까?

**답:**

Layer를 추가하면 더 복잡한 관계를 학습할 수 있지만, 학습이 어려워지거나 과적합이 발생할 수도 있다. 실제 결과를 Baseline과 비교해야 한다.


### Deep Model: `64 → 32 → 16 → 1`

In [ ]:
# Deep Model 생성: 64 → 32 → 16 → 1
deep_model = keras.Sequential([
    # 첫 번째 은닉층
    keras.layers.Dense(64, activation="relu", input_shape=(x_train.shape[1],)),

    # 두 번째 은닉층
    keras.layers.Dense(32, activation="relu"),

    # 세 번째 은닉층 추가
    keras.layers.Dense(16, activation="relu"),

    # 출력층
    keras.layers.Dense(1)
])

# 모델 설정
deep_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 모델 학습
deep_history = deep_model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# 예측
deep_pred = deep_model.predict(x_test, verbose=0).flatten()

# 성능 확인
print("Deep Model :", evaluate(y_test, deep_pred))

### Q25. 더 넓거나 더 깊은 Neural Network가 항상 더 좋은 성능을 보이는가?

### 생각해보기
- Network가 커지면 어떤 관계까지 표현할 수 있을까?
- 반대로 어떤 문제가 생길 수 있을까?

**답:**

항상 더 좋은 것은 아니다. 모델이 넓거나 깊어지면 복잡한 관계를 표현할 수 있지만 파라미터가 많아져 과적합이 발생하거나 학습 시간이 길어질 수 있다.


# STEP 31. Keras Hyperparameter 변경

### Q26. Batch Size를 변경하면 결과가 달라질까?

비교: `32`, `64`, `128`

**답:**

Batch Size가 달라지면 한 번에 학습하는 데이터 수가 달라져 학습 과정과 최종 성능이 달라질 수 있다. 32, 64, 128을 같은 구조에서 비교한다.


In [ ]:
# 비교할 Batch Size
batch_list = [32, 64, 128]

# Batch Size별 Keras 성능 비교
for batch in batch_list:
    # 같은 구조의 모델 생성
    batch_model = keras.Sequential([
        keras.layers.Dense(64, activation="relu", input_shape=(x_train.shape[1],)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1)
    ])

    # 모델 설정
    batch_model.compile(optimizer="adam", loss="mse")

    # 모델 학습
    batch_model.fit(
        x_train, y_train,
        epochs=50,
        batch_size=batch,
        validation_split=0.2,
        verbose=0
    )

    # 예측
    pred = batch_model.predict(x_test, verbose=0).flatten()

    # 결과 출력
    print("batch_size =", batch, evaluate(y_test, pred))

### Q27. Dropout을 적용하면 결과는 어떻게 달라질까?

### 생각해보기
- Train 성능은 어떻게 변할까?
- Train 성능이 조금 낮아져도 Test 성능이 좋아질 수 있을까?

**답:**

Dropout을 사용하면 학습 중 일부 뉴런을 사용하지 않아 Train 성능은 조금 낮아질 수 있다. 하지만 특정 Train 데이터에 너무 맞춰지는 것을 줄여 Test 성능이 좋아질 수도 있다.


In [ ]:
# Dropout을 적용한 모델 생성
dropout_model = keras.Sequential([
    # 첫 번째 은닉층
    keras.layers.Dense(64, activation="relu", input_shape=(x_train.shape[1],)),

    # 뉴런의 20%를 학습 중 랜덤하게 사용하지 않음
    keras.layers.Dropout(0.2),

    # 두 번째 은닉층
    keras.layers.Dense(32, activation="relu"),

    # 출력층
    keras.layers.Dense(1)
])

# 모델 설정
dropout_model.compile(optimizer="adam", loss="mse")

# 모델 학습
dropout_model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# 예측
dropout_pred = dropout_model.predict(x_test, verbose=0).flatten()

# 성능 확인
print("Dropout Model :", evaluate(y_test, dropout_pred))

### Q28. Learning Rate를 변경하면 학습 결과는 어떻게 달라질까?

기본 Adam과 `learning_rate=0.0005`를 비교해보자.

**답:**

Learning Rate는 한 번 학습할 때 가중치를 얼마나 크게 변경할지 정한다. 값이 작으면 천천히 학습하고, 너무 크면 최적의 값을 지나칠 수 있다. 기본 Adam과 0.0005를 비교한다.


In [ ]:
# Learning Rate를 0.0005로 설정한 Adam 생성
optimizer = keras.optimizers.Adam(learning_rate=0.0005)

# 모델 생성
lr_model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(x_train.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1)
])

# 변경한 Adam을 사용하여 모델 설정
lr_model.compile(
    optimizer=optimizer,
    loss="mse"
)

# 모델 학습
lr_model.fit(
    x_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

# 예측
lr_pred_keras = lr_model.predict(x_test, verbose=0).flatten()

# 성능 확인
print("Learning Rate 0.0005 :", evaluate(y_test, lr_pred_keras))

# STEP 32. 실험 결과 정리 및 비교

지금까지 수행한 실험 결과를 별도 표 또는 Excel에 계속 기록하고, Notebook에서도 주요 결과를 하나의 표로 정리합니다.

In [ ]:
# 지금까지 주요 모델 결과를 표로 정리
results = pd.DataFrame([
    ["Linear Regression", *evaluate(y_test, y_pred_lr).values()],
    ["Decision Tree", *evaluate(y_test, y_pred_dt).values()],
    ["Random Forest", *evaluate(y_test, y_pred_rf).values()],
    ["Gradient Boosting", *evaluate(y_test, y_pred_gb).values()],
    ["Keras Baseline", *evaluate(y_test, y_pred_keras).values()],
    ["Keras Wide", *evaluate(y_test, wide_pred).values()],
    ["Keras Deep", *evaluate(y_test, deep_pred).values()],
    ["Keras Dropout", *evaluate(y_test, dropout_pred).values()],
    ["Keras LR 0.0005", *evaluate(y_test, lr_pred_keras).values()]
], columns=["Model", "MAE", "RMSE", "R2"])

# RMSE가 작은 순서로 확인
results.sort_values("RMSE")

### 결과 저장

In [ ]:
# 실험 결과를 CSV 파일로 저장
results.to_csv("DAY2_model_results.csv", index=False)

### Q29. 가장 큰 성능 개선이 나타난 실험은 무엇인가?

**답:**

`results.sort_values('RMSE')` 결과와 DAY1 결과를 비교하여 RMSE가 가장 많이 감소하거나 R²가 가장 많이 증가한 실험을 작성한다.


### Q30. 거의 효과가 없었던 변경은 무엇인가?

**답:**

결과값이 거의 변하지 않은 실험을 작성한다. 일반적으로 Tree 계열 모델에서 Scaler만 변경한 실험은 변화가 작을 가능성이 높다.


### Q31. 가장 복잡한 모델이 가장 좋은 성능을 보였는가?

**답:**

반드시 그렇지는 않다. Wide/Deep 모델의 결과와 다른 모델의 결과를 비교해 가장 복잡한 모델이 실제로 가장 좋은지 확인한다.


### Q32. 이번 데이터에서는 Feature Engineering, Encoding, Scaling, 모델 변경, Hyperparameter, Neural Network 구조 중 어떤 방법의 영향이 컸는가?

**답:**

각 실험의 MAE, RMSE, R² 변화량을 비교하여 판단한다. 한 가지 방법이 항상 가장 큰 영향을 주는 것은 아니며 이번 데이터의 실제 결과를 기준으로 작성한다.


# STEP 33. Overfitting 확인

In [ ]:
# Keras Baseline의 Train Loss와 Validation Loss 그리기
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

# 그래프 제목과 범례
plt.title("Train Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Q33. Train Loss는 계속 감소하지만 Validation Loss가 다시 증가한다면 무엇을 의미할까?

**답:**

Train Loss는 계속 줄어드는데 Validation Loss가 다시 증가하면 Train 데이터에 너무 맞춰지고 있다는 뜻이다. 이를 과적합(Overfitting)이라고 한다.


### Q34. 이 경우 Epoch을 계속 늘리는 것이 좋을까?

**답:**

계속 늘리는 것이 항상 좋지는 않다. Validation Loss가 더 이상 좋아지지 않는다면 학습을 멈추는 EarlyStopping 등을 사용할 수 있다.


# STEP 34. Feature Importance 분석

In [ ]:
# Feature Importance를 확인할 Random Forest 모델 생성
rf_importance = RandomForestRegressor(random_state=1002)

# 모델 학습
rf_importance.fit(X_train, y_train)

# Feature 이름과 중요도를 표로 만들기
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_importance.feature_importances_
})

# 중요도가 높은 순서로 정렬
importance_df = importance_df.sort_values("Importance", ascending=False)

# 상위 15개 확인
importance_df.head(15)

In [ ]:
# Feature Importance 상위 15개 시각화
top15 = importance_df.head(15)

plt.figure(figsize=(8, 6))
plt.barh(top15["Feature"], top15["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance Top 15")
plt.xlabel("Importance")
plt.show()

### Q35. DAY 2에서 새로 만든 Feature는 실제 모델에서 중요하게 사용되었는가?

**답:**

아래 코드에서 `morning_rush`, `evening_rush`, `rain_rush`, `hour_sin`, `hour_cos`의 중요도를 확인한다. 값이 상대적으로 크다면 모델에서 중요하게 사용된 것이다.


In [ ]:
# DAY2에서 새로 만든 Feature만 중요도 확인
new_features = [
    "morning_rush",
    "evening_rush",
    "rain_rush",
    "hour_sin",
    "hour_cos"
]

# 새 Feature의 중요도 출력
importance_df[importance_df["Feature"].isin(new_features)]

### Q36. 새 Feature의 중요도가 낮다면 Feature Engineering이 실패했다고 볼 수 있을까?

**답:**

아니다. 중요도가 낮다고 무조건 실패한 것은 아니다. 다른 Feature와 비슷한 정보를 가지고 있거나 특정 상황에서만 도움이 될 수 있다. 최종적으로는 Feature 추가 전후의 Test 성능도 함께 확인해야 한다.


# STEP 35. Actual vs Predicted / Residual 분석

## Residual(잔차)이란?

Residual은 **실제값과 예측값의 차이**입니다.

`Residual = Actual - Predicted`

예를 들어 실제 대여량이 1,000인데 모델이 800으로 예측했다면:

`Residual = 1000 - 800 = +200`

반대로 모델이 1,200으로 예측했다면:

`Residual = 1000 - 1200 = -200`

### Q37. Residual이 양수와 음수라는 것은 각각 무엇을 의미할까?

- Residual > 0 :
- Residual < 0 :
- Residual = 0 :

**답:**

- Residual > 0 : 실제값이 예측값보다 크다. 모델이 실제보다 작게 예측했다.
- Residual < 0 : 실제값이 예측값보다 작다. 모델이 실제보다 크게 예측했다.
- Residual = 0 : 실제값과 예측값이 같다.


### Q38. Residual과 Absolute Error는 무엇이 다를까?

`Residual = Actual - Predicted`

`Absolute Error = |Actual - Predicted|`

**답:**

Residual은 `실제값 - 예측값`이라서 양수/음수 방향을 알 수 있다. Absolute Error는 절댓값을 사용하므로 방향은 없어지고 오차의 크기만 확인할 수 있다.


### 최종 후보 모델 선택

STEP 32의 결과를 보고 분석할 모델의 예측값을 `final_pred`에 지정합니다.

In [ ]:
# 예시: Random Forest를 최종 후보 모델로 선택
# 다른 모델이 더 좋게 나왔다면 해당 모델의 예측값으로 바꾸면 됨
final_pred = y_pred_rf

# 최종 후보 모델 성능 확인
evaluate(y_test, final_pred)

### Q39. 실제값과 예측값은 얼마나 비슷한가?

**답:**

아래 Actual vs Predicted 그래프에서 점들이 대각선에 가까울수록 실제값과 예측값이 비슷하다. 그래프와 MAE/RMSE/R²를 함께 보고 판단한다.


In [ ]:
# 실제값과 예측값 비교
plt.figure(figsize=(6, 6))
plt.scatter(y_test, final_pred, alpha=0.5)

# 실제값 = 예측값인 기준선
min_value = min(y_test.min(), final_pred.min())
max_value = max(y_test.max(), final_pred.max())
plt.plot([min_value, max_value], [min_value, max_value])

# 그래프 꾸미기
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

### Q40. Residual 분포를 보고 예측 오차를 해석해보자.

### 생각해보기
- Residual은 0을 중심으로 분포하는가?
- 양수 또는 음수 한쪽으로 치우치는가?
- 큰 Residual이 많이 나타나는가?

**답:**

Residual이 0 근처에 많이 모이면 예측이 비교적 잘 된 것이다. 한쪽으로 치우치면 모델이 실제값보다 계속 크게 또는 작게 예측하는 경향이 있을 수 있다. 0에서 아주 먼 값은 크게 틀린 데이터이다.


In [ ]:
# Residual 계산: 실제값 - 예측값
residual = y_test.values - final_pred

# Residual 분포 확인
plt.hist(residual, bins=30)
plt.axvline(0)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Residual Distribution")
plt.show()

# STEP 36. 오차가 큰 데이터 분석

In [ ]:
# Test 데이터 원본을 복사하여 분석용 데이터 만들기
error_df = df.loc[X_test.index].copy()

# 실제값 저장
error_df["actual"] = y_test

# 예측값 저장
error_df["predicted"] = final_pred

# Residual 계산
error_df["residual"] = error_df["actual"] - error_df["predicted"]

# Absolute Error 계산
error_df["abs_error"] = np.abs(error_df["residual"])

# 오차가 큰 데이터부터 확인
error_df.sort_values("abs_error", ascending=False).head(20)

### Q41. 모델이 크게 틀린 데이터에는 어떤 공통점이 있는가?

확인해볼 항목:
- 특정 시간대인가?
- 실제 대여량이 매우 높은가?
- 비가 오는가?
- 특정 계절인가?
- 특정 온도 범위인가?

**답:**

위 표에서 `hour`, `actual`, `is_rain`, 계절, 온도 등의 값을 확인한다. 오차가 큰 상위 데이터에 특정 시간대나 날씨 조건이 반복되는지 찾아보고 실제 결과를 작성한다.


### Q42. 시간대별 평균 예측 오차는 어떻게 다른가?

**답:**

아래 결과에서 시간대별 `abs_error` 평균을 비교한다. 평균값이 큰 시간대일수록 모델이 해당 시간대를 상대적으로 잘 예측하지 못한 것이다.


In [ ]:
# 시간대별 평균 Absolute Error 계산
hour_error = error_df.groupby("hour")["abs_error"].mean()

# 결과 확인
print(hour_error)

# 시간대별 평균 오차 시각화
hour_error.plot(kind="bar", figsize=(10, 4))
plt.ylabel("Mean Absolute Error")
plt.title("Mean Absolute Error by Hour")
plt.show()

# STEP 37. 최종 모델 선정

### 최종 모델

모델: `STEP 32 결과에서 RMSE가 작고 R²가 높은 모델 작성`

Feature Engineering: `morning_rush, evening_rush, rain_rush, hour_sin, hour_cos`

Encoding: `실제로 최종 모델에 사용한 Encoding 작성`

Scaling: `실제로 최종 모델에 사용한 Scaler 작성`

변경한 Hyperparameter 또는 Network 구조: `최종 모델에서 변경한 내용 작성`

MAE: `실행 결과 입력`

RMSE: `실행 결과 입력`

R²: `실행 결과 입력`

### 선정 이유

MAE와 RMSE가 작고 R²가 높은 모델을 최종 모델로 선정한다. Train 성능만 보지 않고 Test 성능을 기준으로 비교한다.

### 가장 효과가 있었던 실험

STEP 32의 결과에서 성능 개선 폭이 가장 컸던 실험을 작성한다.

### 효과가 거의 없었던 실험

실험 전후 MAE, RMSE, R² 변화가 거의 없었던 방법을 작성한다.

### 모델이 잘 예측하지 못한 데이터

STEP 36에서 Absolute Error가 큰 데이터의 공통점을 확인하여 작성한다.

### 추가로 개선한다면

다른 Feature를 추가하거나 Hyperparameter를 더 세밀하게 조절하여 성능을 비교해볼 수 있다.


# STEP 38. Wrap-up

### Q43. Feature를 추가했다고 항상 성능이 좋아졌는가?

**답:**

아니다. 새로운 Feature가 유용하면 성능이 좋아질 수 있지만, 필요 없는 Feature이거나 기존 Feature와 비슷한 정보라면 성능이 거의 변하지 않거나 나빠질 수도 있다.


### Q44. Scaler를 변경했을 때 모든 모델의 성능이 달라졌는가?

**답:**

아니다. Tree 계열 모델은 Scaling의 영향이 작은 편이고, Neural Network는 입력값의 범위에 영향을 받을 수 있어 모델마다 결과가 다르게 나타날 수 있다.


### Q45. Tree 계열 모델은 왜 Scaling의 영향을 거의 받지 않았는가?

**답:**

Tree 계열은 값 자체의 크기보다 어떤 값을 기준으로 데이터를 나눌지를 중요하게 본다. Scaling 후에도 값의 순서가 유지되므로 결과가 거의 변하지 않을 수 있다.


### Q46. Encoding 방법에 따라 성능이 달라질 수 있는 이유는 무엇인가?

**답:**

Encoding 방법에 따라 모델에 들어가는 숫자의 의미와 Feature 개수가 달라지기 때문이다. 특히 Ordinal Encoding은 순서가 없는 범주에도 숫자의 순서를 만들 수 있다.


### Q47. Hyperparameter를 변경했다고 항상 성능이 개선되었는가?

**답:**

아니다. Hyperparameter가 데이터에 잘 맞으면 좋아질 수 있지만 너무 크거나 작은 값은 오히려 성능을 떨어뜨리거나 과적합을 만들 수 있다.


### Q48. Neural Network를 더 넓거나 깊게 만들면 항상 성능이 좋아졌는가?

**답:**

아니다. 더 넓거나 깊은 모델은 복잡한 관계를 학습할 수 있지만 과적합이 발생하거나 학습이 어려워져 Test 성능이 나빠질 수도 있다.


### Q49. 이번 실습에서 모델 성능을 개선하기 위해 시도한 방법을 정리해보자.

1.

2.

3.

4.

5.

**답:**

1. 출퇴근 시간, 비 오는 출퇴근 시간, 시간 주기 Feature 추가

2. StandardScaler, MinMaxScaler, RobustScaler 비교

3. One-Hot Encoding과 Ordinal Encoding 비교

4. Random Forest와 Gradient Boosting의 Hyperparameter 변경

5. Keras의 Hidden Unit, Layer, Batch Size, Dropout, Learning Rate 변경
